## ATIVIDADE 1 - Sintonizando o PID

In [ ]:
import numpy as np

# Parameters (EC5)
m = 0.073 # (Kg)
g = 9.81 #(m/s^2)
k = 6.51e-5 #(Nm^2/A^2) 
x0 = 8.5e-3 #(m)
i0 = np.sqrt(2*m*g*x0*x0/k) #(A)

In [ ]:
# Resistance
R = 11.0 #(Ohm)
u0 = R*i0

In [ ]:
eps = 0.029*x0
Kp = 1800
Ki = 500
Kd = 180

In [ ]:
# Magnetic force fm
def fm(i,x):
    return k/2*i*i/x/x
# Force Factor
def Bl(i,x):
     return k/2*i/x/x #(N/A)
# Inductance
def L(x):
     return k/x #(H)

def odePID(t,y):
    i,x,v,E = y
    e = x-x0
    u = u0 + Kp*e + Ki*E + Kd*v
    
    #di/dt
    di = -R/L(x)*i - Bl(i,x)/L(x)*v + u/L(x)
    #dx/dt
    dx = v
    #dv/dt
    dv = g-fm(i,x)/m
    #dE/dt
    dE = e
   
    return [di, dx, dv, dE]

print("\nAt equilibrium:")
print("    i0 = ",i0)
print("    x0 = ",x0)
print("    fm = ",fm(i0,x0))
print("    mg = ",m*g)
print("    fm/fg (%) = ",fm(i0,x0)/(m*g)*100)

In [ ]:
from scipy.integrate import solve_ivp

# Initial conditions
# y = [i,x,v,E]
y_0 = [i0,x0 + eps,0.0,0.0]
# Time span
t_0 = 0.0
t_end = 30 #(s) 

sol = solve_ivp(odePID, [t_0, t_end],y_0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

i = sol.y[0,:]
x = sol.y[1,:]
v = sol.y[2,:]
E = sol.y[3,:]

e = x - x0
u = u0 + Kp*e + Ki*E + Kd*v

# --- Position plot --- 
fig = plt.figure(figsize=(15,4))

plt.subplot(1, 2, 1)
plt.plot(sol.t, x - x0, 'k', linewidth=2)
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$x(t)-x_0$ (m)')
fig.suptitle('Fig. 1 - Dados da esfera')

plt.subplot(1, 2, 2)
plt.plot(sol.t, v, 'k', linewidth=2)
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$v(t)$ (m/s)')

# --- Source plot ---
fig = plt.figure(figsize=(15,4))
plt.plot(sol.t, i, 'k', linewidth=2, label='$i(t)$')
plt.axhline(2.0, color='r', linestyle='--', linewidth=1.5, label='Limite de 2 A')
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$i(t)$ (A)')
plt.legend()
fig.suptitle('Fig. 2 - Dados da fonte')

# --- Ball position plot ---
fig = plt.figure(figsize=(15,4))
plt.plot(sol.t, x, 'k', linewidth=2, label='$x(t)$')
plt.axhline(x0, color='r', linestyle='--', linewidth=1.5, label='$x_0$')
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$x(t)$ (m)')
plt.legend()
plt.ticklabel_format(axis='y', style='plain', useOffset=False)
fig.suptitle('Fig. 3 - Posição da bola')

# --- Control voltage plot ---
fig = plt.figure(figsize=(15,4))
plt.plot(sol.t, u, 'k', linewidth=2)
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$u(t)$ (V)')
fig.suptitle('Fig. 4 - Tensão de controle $u(t)$')

plt.show()

print("eps =", eps)
print("x(0) =", y_0[1])
print("x_final =", x[-1])
print("x_final - x0 =", x[-1] - x0)
print("v_final =", v[-1])
print("i_max =", np.max(i))
print("u_max =", np.max(u))
print("u_min =", np.min(u))

## ATIVIDADE 2 - Adicionando incerteza à malha de realimentação

In [ ]:
import numpy as np

# Parameters (EC5)
m = 0.073 # (Kg)
g = 9.81 #(m/s^2)
k = 6.51e-5 #(Nm^2/A^2) 
x0 = 8.5e-3 #(m)
i0 = np.sqrt(2*m*g*x0*x0/k) #(A)

In [ ]:
# Resistance
R = 11.0 #(Ohm)
u0 = R*i0

In [ ]:
eps = 0.029*x0
Kp = 1800
Ki = 500
Kd = 180

In [ ]:
import numpy as np

# Magnetic force fm
def fm(i,x):
    return k/2*i*i/x/x
# Force Factor
def Bl(i,x):
     return k/2*i/x/x #(N/A)
# Inductance
def L(x):
     return k/x #(H)

def odePID(t,y):
    i,x,v,E = y
    noise = np.random.normal(0, 0.03 * x0)
    x_noise = x + noise 
    e = x_noise-x0
    u = u0 + Kp*e + Ki*E + Kd*v
    
    #di/dt
    di = -R/L(x)*i - Bl(i,x)/L(x)*v + u/L(x)
    #dx/dt
    dx = v
    #dv/dt
    dv = g-fm(i,x)/m
    #dE/dt
    dE = e
   
    return [di, dx, dv, dE]

In [ ]:
# Initial conditions
# y = [i,x,v,E]
y_0 = [i0,x0 + eps,0.0,0.0]
# Time span
t_0 = 0.0
t_end = 30 #(s) 

sol = solve_ivp(odePID, [t_0, t_end],y_0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

i = sol.y[0,:]
x = sol.y[1,:]
v = sol.y[2,:]
E = sol.y[3,:]

e = x - x0
u = u0 + Kp*e + Ki*E + Kd*v

# --- Position plot --- 
fig = plt.figure(figsize=(15,4))

plt.subplot(1, 2, 1)
plt.plot(sol.t, x - x0, 'k', linewidth=2)
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$x(t)-x_0$ (m)')
fig.suptitle('Fig. 1 - Dados da esfera')

plt.subplot(1, 2, 2)
plt.plot(sol.t, v, 'k', linewidth=2)
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$v(t)$ (m/s)')

# --- Source plot ---
fig = plt.figure(figsize=(15,4))
plt.plot(sol.t, i, 'k', linewidth=2, label='$i(t)$')
plt.axhline(2.0, color='r', linestyle='--', linewidth=1.5, label='Limite de 2 A')
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$i(t)$ (A)')
plt.legend()
fig.suptitle('Fig. 2 - Dados da fonte')

# --- Ball position plot ---
fig = plt.figure(figsize=(15,4))
plt.plot(sol.t, x, 'k', linewidth=2, label='$x(t)$')
plt.axhline(x0, color='r', linestyle='--', linewidth=1.5, label='$x_0$')
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$x(t)$ (m)')
plt.legend()
plt.ticklabel_format(axis='y', style='plain', useOffset=False)
fig.suptitle('Fig. 3 - Posição da bola')

# --- Control voltage plot ---
fig = plt.figure(figsize=(15,4))
plt.plot(sol.t, u, 'k', linewidth=2)
plt.grid(True)
plt.xlabel('$t$ (s)')
plt.ylabel('$u(t)$ (V)')
fig.suptitle('Fig. 4 - Tensão de controle $u(t)$')

plt.show()